# Visualize an MITgcm cs32x15 global-ocean run

Reads the Zarr written by `datagen.mitgcm.global_ocean.scripts.run` or
`datagen.mitgcm.global_ocean.scripts.preflight` and renders six-face panels,
a cube-unfolded view, and time series of basin-mean diagnostics.

Dataset layout: `(time, field, face, y, x)` with `face=6, y=x=32` and per-cell
longitude/latitude coords `xc(face, y, x)` / `yc(face, y, x)`.

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`. All ship with the `datagen`
project env.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

zarr_override = os.environ.get("GLOBAL_OCEAN_ZARR")
if zarr_override:
    DATA_PATH = Path(zarr_override)
else:
    DATA_ROOT = Path(os.environ.get(
        "DATA_ROOT",
        f"{os.environ.get('SCRATCH', '/tmp')}/fots-data/mitgcm/global-ocean",
    ))
    CORNER = os.environ.get("CORNER", "corner_00")
    DATA_PATH = DATA_ROOT / "preflight" / CORNER / "run.zarr"

ds = xr.open_zarr(DATA_PATH)
ds

In [ ]:
print("Path:", DATA_PATH)
print("Dims:", dict(ds.sizes))
print("Time range:", float(ds.time.min()) / 86400, "..",
      float(ds.time.max()) / 86400, "days")
print("Lon (xc) range:", float(ds.xc.min()), "..", float(ds.xc.max()))
print("Lat (yc) range:", float(ds.yc.min()), "..", float(ds.yc.max()))
print("Fields:    ", [str(f) for f in ds.field.values])
print("Run params:", ds.attrs.get("params"))

## Six-face snapshot

First saved state from the run. Each row is one field; columns 1–6 show the
six cube faces in `face` order (1 → 6).

In [ ]:
n_face = ds.sizes["face"]
field_names = [str(f) for f in ds.field.values]

def field_at(name, t):
    return ds.data.sel(field=name).isel(time=t).values

def show_panel(ax, arr, title, *, cmap, vmin=None, vmax=None):
    finite = np.isfinite(arr)
    if not finite.any():
        ax.text(0.5, 0.5, "NaN", ha="center", va="center", transform=ax.transAxes)
    else:
        if vmin is None or vmax is None:
            vmin = float(np.nanmin(arr))
            vmax = float(np.nanmax(arr))
        ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax,
                  aspect="equal")
    ax.set_title(title, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

snap = {name: field_at(name, 0) for name in field_names}
fig, axes = plt.subplots(len(field_names), n_face, figsize=(2 * n_face, 2 * len(field_names)),
                         constrained_layout=True)
if len(field_names) == 1:
    axes = axes[np.newaxis, :]

for r, name in enumerate(field_names):
    arr = snap[name]
    cmap = "viridis" if name.startswith(("theta", "salt")) else "RdBu_r"
    if cmap == "RdBu_r":
        m = float(np.nanmax(np.abs(arr)))
        vmin, vmax = -m, m
    else:
        vmin = float(np.nanmin(arr)); vmax = float(np.nanmax(arr))
    for f in range(n_face):
        title = f"{name}\nface {f + 1}" if f == 0 else f"face {f + 1}"
        show_panel(axes[r, f], arr[f], title, cmap=cmap, vmin=vmin, vmax=vmax)

fig.suptitle(f"t = {float(ds.time.isel(time=0)) / 86400:.2f} days")
plt.show()

## Cube-unfolded `eta`

Sadourny-style cross layout: faces 1–4 along the equator, face 5 above
(north pole), face 6 below (south pole).

The exact face↔geographic-region mapping depends on the exch2 default
topology; this layout is a quick visual sanity check rather than a
geographically faithful projection.

In [ ]:
eta_all = ds.data.sel(field="eta").values  # (time, face, y, x)
vmax_eta = float(np.nanmax(np.abs(eta_all)))

def plot_cross(eta, title):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9), constrained_layout=True)
    for ax in axes.flat:
        ax.axis("off")
    # row 0: face 5 above face 1
    axes[0, 0].imshow(eta[4], origin="lower", cmap="RdBu_r",
                      vmin=-vmax_eta, vmax=vmax_eta, aspect="equal")
    axes[0, 0].set_title("face 5")
    # row 1: faces 1 2 3 4
    for col, f in enumerate([0, 1, 2, 3]):
        axes[1, col].imshow(eta[f], origin="lower", cmap="RdBu_r",
                            vmin=-vmax_eta, vmax=vmax_eta, aspect="equal")
        axes[1, col].set_title(f"face {f + 1}")
    # row 2: face 6 below face 1
    axes[2, 0].imshow(eta[5], origin="lower", cmap="RdBu_r",
                      vmin=-vmax_eta, vmax=vmax_eta, aspect="equal")
    axes[2, 0].set_title("face 6")
    fig.suptitle(title)
    return fig

plot_cross(eta_all[0], f"eta at t = {float(ds.time.isel(time=0)) / 86400:.2f} d")
plot_cross(eta_all[-1], f"eta at t = {float(ds.time.isel(time=-1)) / 86400:.2f} d")
plt.show()

## Globally-integrated diagnostics

Cell-area-weighted means use the cosine of latitude as a cheap proxy for
the cubed-sphere cell area. This is exact only on the lat-lon grid; on the
cubed sphere it gives a leading-order estimate sufficient for spotting
drift in a preflight run.

In [ ]:
yc = ds.yc.values  # (face, y, x)
weights = np.cos(np.deg2rad(yc))
weights = weights / weights.sum()

def weighted_mean(arr):
    return (arr * weights).sum(axis=(-3, -2, -1))

theta = ds.data.sel(field="theta_k1").values
salt = ds.data.sel(field="salt_k1").values
u = ds.data.sel(field="u_k2").values
v = ds.data.sel(field="v_k2").values
speed = np.sqrt(u ** 2 + v ** 2)

t_d = ds.time.values / 86400.0

fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
axes[0].plot(t_d, weighted_mean(theta));   axes[0].set_title("<theta_k1> [degC]")
axes[1].plot(t_d, weighted_mean(salt));    axes[1].set_title("<salt_k1> [psu]")
axes[2].plot(t_d, weighted_mean(eta_all)); axes[2].set_title("<eta> [m]")
axes[3].plot(t_d, speed.max(axis=(-3, -2, -1))); axes[3].set_title("max speed_k2 [m/s]")
for ax in axes:
    ax.set_xlabel("time [days]")
plt.show()